# Deploying NVIDIA Nemotron-3-Ultra with vLLM

This notebook will walk you through how to run the `nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B` model with vLLM.

[vLLM](https://docs.vllm.ai) is a fast and easy-to-use library for LLM inference and serving.

For more details on the model [click here](https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-BF16)

Prerequisites:
- NVIDIA GPUs with recent drivers and CUDA 12.x
  - BF16: 8x GB200/B200/B300, 16x H100, 8x H200 | NVFP4: 4x GB200/B200/B300/GB300, 8x H100
- [Docker](https://docs.docker.com/engine/install/) with [NVIDIA Container Toolkit](https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/install-guide.html)
- Python 3.10+

## Overview

- **Serve** the Nemotron-3-Ultra model using vLLM
- **Query the model** through an OpenAI-compatible REST API
- **Invoke tools** using structured function-calling outputs
- **Tune reasoning depth** by configuring the model's thinking budget

## Table of Contents

1. **Environment setup** - Dependencies and virtual environment
2. **Verify GPU** - Confirm CUDA and GPU availability
3. **OpenAI-compatible server** - Launch and query vLLM
   - **Start server** - BF16 and NVFP4 variants
   - **Generate responses** - Single, batched, and streamed completions
   - **Reasoning** - Toggle thinking on/off
   - **Tool calling** - Function calling via OpenAI tools schema
   - **Controlling Reasoning Budget** - Limit reasoning trace length
4. **Cleanup and shutdown**

#### Launch on NVIDIA Brev
You can simplify the environment setup by using [NVIDIA Brev](https://developer.nvidia.com/brev). For the easiest way to get started, check out the NVFP4 variant on a Brev instance with the necessary dependencies pre-configured.

Once deployed, click on the "Open Notebook" button to get started with this guide.

**For NVFP4 (8xH100):**

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-3EPQRUP8Sl27sxp1fMvXt3Lor8T)

## Environment setup

### Pull the vLLM Docker image

The model runs inside the official vLLM container. Pull it once before starting the server:

```shell
docker pull vllm/vllm-openai:v0.22.0
```

### Install notebook client dependencies

The cells below only need the OpenAI client and tokenizer libraries — vLLM itself runs inside Docker.

In [ ]:
# Ensure pip is available
!python3 -m ensurepip --default-pip

In [ ]:
%pip install -U openai==2.38.0 transformers==5.9.0 torch==2.11.0 --extra-index-url https://download.pytorch.org/whl/cu130

## Verify GPU

Confirm your GPUs are visible on the host before starting the Docker container.

> **Expected output:** 8 GPUs listed with driver and CUDA version shown. If no GPUs appear, check your driver installation.

In [ ]:
# GPU environment check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Num GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU[{i}]: {torch.cuda.get_device_name(i)}")

CUDA available: True
Num GPUs: 8
GPU[0]: NVIDIA B200
GPU[1]: NVIDIA B200
GPU[2]: NVIDIA B200
GPU[3]: NVIDIA B200
GPU[4]: NVIDIA B200
GPU[5]: NVIDIA B200
GPU[6]: NVIDIA B200
GPU[7]: NVIDIA B200


## OpenAI-compatible server

Serve the model via an OpenAI-compatible API using vLLM.

### Launch the Docker container

Open a terminal on the host and start an interactive shell inside the vLLM container. The `--network=host` flag makes the server reachable at `localhost:8000` from the notebook.

```shell
docker run --rm -it --gpus all --ipc=host --network=host \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  --entrypoint /bin/bash \
  vllm/vllm-openai:v0.22.0
```

> **Note:** Mount the HuggingFace cache directory so model weights are read from disk rather than re-downloaded on each run. Replace `~/.cache/huggingface` if your cache is in a different location.

All `vllm serve` commands below should be run from inside this container.

### Configuration reference

| | BF16 | NVFP4 |
|---|---|---|
| **Model** | `nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-BF16` | `nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-NVFP4` |
| **Supported hardware** | 8x GB200/B200/B300, 16x H100, 8x H200 | 4x GB200/B200/B300/GB300, 8x H100 |
| **Docker image** | `vllm/vllm-openai:v0.22.0` | `vllm/vllm-openai:v0.22.0` |
| **TP / DP (this notebook)** | 8 / 1 | 1 / 8 |
| **Reasoning parser** | `nemotron_v3` | `nemotron_v3` |
| **Tool parser** | `qwen3_coder` | `qwen3_coder` |
| **Port** | 8000 | 8000 |

### Start server

Run one of the following commands from inside the Docker container terminal.

> **Note:** Parser names are backend-specific and not interchangeable. vLLM uses `--reasoning-parser nemotron_v3` and `--tool-call-parser qwen3_coder`. TensorRT-LLM uses `--reasoning_parser nano-v3` and SGLang uses `--reasoning-parser nemotron_3` — these are different identifiers for the same logical capability.

#### BF16 (8x B200)

```shell
vllm serve nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-BF16 \
  --served-model-name nemotron-3-ultra \
  --trust-remote-code \
  --tensor-parallel-size 8 \
  --kv-cache-dtype fp8 \
  --max-model-len 262144 \
  --gpu-memory-utilization 0.90 \
  --enable-expert-parallel \
  --max-num-seqs 16 \
  --max-num-batched-tokens 32768 \
  --async-scheduling \
  --enable-flashinfer-autotune \
  --mamba-backend triton \
  --mamba-ssm-cache-dtype float32 \
  --tool-call-parser qwen3_coder \
  --reasoning-parser nemotron_v3 \
  --host 0.0.0.0 \
  --port 8000
```

#### NVFP4 (8x H100)

```shell
vllm serve nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-NVFP4 \
  --served-model-name nemotron-3-ultra \
  --trust-remote-code \
  --tensor-parallel-size 1 \
  --data-parallel-size 8 \
  --mamba-cache-dtype float16 \
  --mamba-ssm-cache-dtype float16 \
  --mamba-cache-philox-rounds 5 \
  --enable-mamba-cache-stochastic-rounding \
  --max-num-seqs 128 \
  --tool-call-parser qwen3_coder \
  --reasoning-parser nemotron_v3 \
  --host 0.0.0.0 \
  --port 8000
```

#### Optional performance flags

**MTP (Multi-Token Prediction):** May improve throughput in some cases. Add the following flags to either serve command:

```
--speculative_config.method mtp --speculative_config.num_speculative_tokens 5
```

**FP4 KV cache (Blackwell only):** Reduces KV cache memory footprint on Blackwell GPUs. Add the following flag to the BF16 serve command:

```
--kv-cache-dtype nvfp4
```

### Wait for the server to be ready

Before sending any requests, confirm the server is up by polling `/v1/models`.

Run the following in the same terminal (or a separate one):

```shell
until curl -sf http://localhost:8000/v1/models > /dev/null 2>&1; do
  echo "Waiting for server..."; sleep 5
done
echo "Server is ready"
```

> **Expected output:** Once the server is ready, the loop exits and prints `Server is ready`. You can also run `curl http://localhost:8000/v1/models` directly to see the list of loaded model IDs.

> **Note:** On the first run, the model weights will be downloaded from Hugging Face before loading begins. For a large model like this, the combined download and load time can be significant.

### Generate responses

The cells below show single, batched, and streamed completions, followed by reasoning on/off, tool calling, and reasoning budget examples.

In [1]:
from openai import OpenAI

# Set this to match the --served-model-name used when starting the server
SERVED_MODEL_NAME = "nemotron-3-ultra" 
BASE_URL = "http://127.0.0.1:8000/v1"

client = OpenAI(base_url=BASE_URL, api_key="null")

In [2]:
# Single chat completion
response = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Briefly explain: what is vLLM and why is it useful for large model inference?"},
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=512,
)
print(response.choices[0].message.content)

**vLLM** is an open-source library for **high-throughput, memory-efficient LLM inference and serving**.

### Core Innovation: **PagedAttention**
Traditional inference engines reserve a contiguous block of GPU memory (KV cache) for the *maximum* possible sequence length for every request. This causes massive **internal fragmentation** (wasted memory) because actual sequences are usually much shorter.

**PagedAttention** solves this by treating the KV cache like OS virtual memory:
*   It splits the KV cache into fixed-size **blocks (pages)**.
*   Blocks are allocated **on-demand** and can be **non-contiguous** in physical GPU memory.
*   This enables **near-zero memory waste** and allows the same physical blocks to be shared across requests (e.g., for parallel sampling or prefix sharing in RAG).

---

### Why It Is Useful (Key Benefits)

| Feature | Impact |
| :--- | :--- |
| **2–4x Higher Throughput** | Efficient memory management allows much larger batch sizes (more concurrent users) o

### Batched completions

Send multiple prompts in sequence and collect all responses.

In [3]:
prompts = [
    "What is the square root of 144?",
    "What is the capital of France?",
    "Explain quantum computing in simple terms.",
]

for prompt in prompts:
    response = client.chat.completions.create(
        model=SERVED_MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=1.0,
        top_p=0.95,
        max_tokens=256,
    )
    print(f"Q: {prompt}")
    print(f"A: {response.choices[0].message.content}\n")

Q: What is the square root of 144?
A: The square root of 144 is **12**.

Q: What is the capital of France?
A: The capital of France is **Paris**.

Q: Explain quantum computing in simple terms.
A: Imagine you are in a giant library looking for a single specific book.

**A classical computer (the one you're using now)** is like a librarian who checks the books **one by one**, very fast. It looks at book 1, then book 2, then book 3... until it finds the right one.

**A quantum computer** is like a magical librarian who can look at **all the books at the exact same time**.

Here is the simple breakdown of how it works:

---

### 1. The Bit vs. The Qubit
*   **Classical Bit:** A light switch. It is either **ON (1)** or **OFF (0)**.
*   **Quantum Bit (Qubit):** A spinning coin. While it’s spinning, is it Heads or Tails? It’s actually a blur of **both at the same time**. Only when you stop it (measure it) does it become one or the other.

This ability to be both 0 and 1 simultaneously is call

### Streamed generation

Receive tokens as they are generated using the OpenAI streaming API.

In [4]:
# Streaming chat completion
stream = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta
    if delta and delta.content:
        print(delta.content, end="", flush=True)

The first 5 prime numbers are:

**2, 3, 5, 7, 11**

### Reasoning

The model supports two modes — **Reasoning ON** (default) and **Reasoning OFF**.

Toggle by setting `enable_thinking` to `False` in `chat_template_kwargs`. Use `temperature=1.0, top_p=0.95` when reasoning is on, and `temperature=0.2` when it is off.

In [5]:
# Reasoning on (default)
print("Reasoning on")
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
)
print("Reasoning:", resp.choices[0].message.reasoning)
print("Content:", resp.choices[0].message.content)
print()

# Reasoning off
print("Reasoning off")
resp2 = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 interesting facts about vLLM."}
    ],
    temperature=0.2,
    max_tokens=256,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)
print(resp2.choices[0].message.content)

Reasoning on
Reasoning: The user wants a haiku about GPUs.
A haiku has a 5-7-5 syllable structure.
I need to think of imagery related to GPUs: parallel processing, graphics, gaming, AI, heat, fans, cores.
Content: Thousands of cores,
Parallel dreams rendered fast,
Pixels bloom to life.

Reasoning off
Here are 3 interesting facts about vLLM:

### 1. It solves the "Memory Fragmentation" problem with PagedAttention
Traditional LLM serving frameworks (like Hugging Face Transformers) allocate a contiguous block of memory for the Key-Value (KV) cache for the maximum possible sequence length. This leads to massive **internal fragmentation** (wasted reserved space) and **external fragmentation** (gaps between allocated blocks).

vLLM introduced **PagedAttention**, an algorithm inspired by virtual memory and paging in operating systems. It breaks the KV cache into fixed-size blocks (pages) that can be stored non-contiguously in GPU memory. This allows vLLM to achieve near-optimal memory usage (

### Tool calling

Call functions using the OpenAI Tools schema and inspect the returned `tool_calls`.

In [6]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate_tip",
            "parameters": {
                "type": "object",
                "properties": {
                    "bill_total": {
                        "type": "integer",
                        "description": "The total amount of the bill"
                    },
                    "tip_percentage": {
                        "type": "integer",
                        "description": "The percentage of tip to be applied"
                    }
                },
                "required": ["bill_total", "tip_percentage"]
            }
        }
    }
]

completion = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": ""},
        {"role": "user", "content": "My bill is $50. What will be the amount for 15% tip?"}
    ],
    tools=TOOLS,
    temperature=1.0,
    top_p=0.95,
    max_tokens=512,
    stream=False
)

print("Reasoning:", completion.choices[0].message.reasoning)
print("Tool calls:", completion.choices[0].message.tool_calls)

Reasoning: The user is asking for a 15% tip on a $50 bill. I have a calculate_tip function that takes bill_total and tip_percentage as integers. I need to call it with bill_total=50, tip_percentage=15. Let me do that.
Tool calls: [ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-bb3bdfefe72532de', function=Function(arguments='{"bill_total": 50, "tip_percentage": 15}', name='calculate_tip'), type='function')]


### Controlling Reasoning Budget

The `reasoning_budget` parameter lets you limit how long the model reasons before producing a response. When the reasoning trace reaches the token budget, the model will try to wrap up at the next newline.

> **Note:** If no newline is encountered within 500 tokens after the budget threshold, the reasoning trace is forcibly terminated at `reasoning_budget + 500` tokens.

In [10]:
from typing import Any, Dict, List
import openai
from transformers import AutoTokenizer


class ThinkingBudgetClient:
    def __init__(self, base_url: str, api_key: str, tokenizer_name_or_path: str):
        self.base_url = base_url
        self.api_key = api_key
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name_or_path)
        self.client = openai.OpenAI(base_url=self.base_url, api_key=self.api_key)

    def chat_completion(
        self,
        model: str,
        messages: List[Dict[str, Any]],
        reasoning_budget: int = 512,
        max_tokens: int = 1024,
        **kwargs,
    ) -> Dict[str, Any]:
        assert (
            max_tokens > reasoning_budget
        ), f"reasoning_budget must be smaller than max_tokens. Given {max_tokens=} and {reasoning_budget=}"

        response = self.client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=reasoning_budget,
            **kwargs
        )

        reasoning_content = response.choices[0].message.reasoning or ""

        if "</think>" not in reasoning_content:
            reasoning_content = f"{reasoning_content}.\n</think>\n\n"

        reasoning_tokens_used = len(
            self.tokenizer.encode(reasoning_content, add_special_tokens=False)
        )
        remaining_tokens = max_tokens - reasoning_tokens_used

        assert (
            remaining_tokens > 0
        ), f"remaining tokens must be positive. Given {remaining_tokens=}. Increase max_tokens or lower reasoning_budget."

        messages.append({"role": "assistant", "content": reasoning_content})
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            continue_final_message=True,
        )

        response = self.client.completions.create(
            model=model,
            prompt=prompt,
            max_tokens=remaining_tokens,
            **kwargs
        )

        return {
            "reasoning_content": reasoning_content.strip().strip("</think>").strip(),
            "content": response.choices[0].text,
            "finish_reason": response.choices[0].finish_reason,
        }

In [11]:
budget_client = ThinkingBudgetClient(
    base_url="http://localhost:8000/v1",
    api_key="null",
    tokenizer_name_or_path="nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-BF16"  # use actual HF model ID for tokenizer
)

In [13]:
resp = budget_client.chat_completion(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    max_tokens=256,
    reasoning_budget=64
)
print("Reasoning:", resp["reasoning_content"])
print("Content:", resp["content"])

Reasoning: The user wants a haiku about GPUs.
A haiku has a 5-7-5 syllable structure.
Topic: Graphics Processing Units, parallel processing, gaming, AI, rendering..
Content: Silicon canvas,
Thousands of cores calculate,
Pixels bloom to life.


## Cleanup and shutdown

To free resources after this notebook:

1. Stop the Docker container in the terminal where it was started (`Ctrl+C`). The `--rm` flag automatically removes the container on exit.
2. Restart the kernel if needed to ensure a clean state.

In [ ]:
print("Kernel cleared. Stop the Docker container in the terminal with Ctrl+C.")